# Computer Lab: Dispersion and Chromaticity — local Python version

This notebook replaces the original Sirepo/Elegant workflow with a local Xsuite-based workflow. The helper module `uspas_labs.dispersion_chromaticity` is installed from the course GitHub repo and contains the first-order dispersion model used in Sections A and B, plus plotting and widget wrappers. Section C uses Xsuite directly for ring tunes, chromaticity, and off-momentum tune scans. The notebook cells expose the main Xsuite construction and physics operations students should inspect.

The examples are intentionally pedagogical: drifts, thick quadrupoles, sector bends, linear dispersion, and tune-footprint plots. They are not a production lattice design.

<style>
.answer {
    color: #b00020;
    border-left: 4px solid #b00020;
    padding-left: 0.8em;
    margin: 0.8em 0;
}
.answer table, .answer th, .answer td {
    color: #b00020;
}
.instructor-note {
    color: #7a3b00;
    border-left: 4px solid #7a3b00;
    padding-left: 0.8em;
    margin: 0.8em 0;
}
.note {
    background-color: #f6f8fa;
    padding: 0.75em;
    border-left: 4px solid #999;
}

.widget-slider, .jupyter-widgets.widget-slider {
    width: 900px;
    max-width: 100%;
}
.widget-slider .noUi-horizontal, .jupyter-widgets.widget-slider .noUi-horizontal {
    height: 12px;
}
.widget-slider .noUi-horizontal .noUi-handle,
.jupyter-widgets.widget-slider .noUi-horizontal .noUi-handle {
    width: 24px;
    height: 24px;
    right: -12px;
    top: -7px;
    border-radius: 50%;
}
.widget-slider .widget-readout, .jupyter-widgets.widget-slider .widget-readout {
    min-width: 70px;
}
</style>


## Learning goals

By the end of this lab you should be able to:

1. Explain how a bend creates horizontal dispersion.
2. Use $\sigma_x^2=\epsilon_x\beta_x+D_x^2\sigma_\delta^2$ to estimate beam-size growth from momentum spread.
3. Tune a two-bend section to make an achromat.
4. Distinguish local zero dispersion from globally small dispersion.
5. Estimate momentum acceptance from both aperture and chromatic resonance crossing.
6. Interpret a tune footprint on a low-order resonance diagram.


In [ ]:
from pathlib import Path
import sys

try:
    import xtrack as xt
    print("xtrack is already installed.")
except ImportError:
    print("Installing xtrack...")
    %pip install --upgrade xsuite
    import xtrack as xt

LOCAL_DEV = Path("uspas_labs").exists()

if LOCAL_DEV:
    print("Using local development version of uspas_labs")
    sys.path.insert(0, str(Path.cwd()))
else:
    # Colab/student path only
    print("Using release version of uspas_labs from GitHub")
    HELPER_VERSION = "main"  # Replace with a release tag, e.g. "v2026-lab1", for student releases.
    HELPER_REPO = "git+https://github.com/r-hensley/uspas-2026-jupyter.git"
    %pip install -q "{HELPER_REPO}@{HELPER_VERSION}"

import numpy as np
import pandas as pd

from IPython.display import HTML, display

import xtrack as xt
from uspas_labs import dispersion_chromaticity as dch
import plotly.io as pio

try:
    from google.colab import output as colab_output # type: ignore
except ImportError:
    print("Not running in Colab, using plotly_mimetype renderer")
    pio.renderers.default = "plotly_mimetype"
    
    # also import autoreload feature
    %load_ext autoreload
    %autoreload 2
else:
    print("Running in Colab, using colab renderer")
    colab_output.enable_custom_widget_manager()
    pio.renderers.default = "colab"

pd.set_option("display.precision", 6)
display(dch.check_environment())


## Working conventions

All distances are in meters. Quadrupole strength $k_1$ is in $\mathrm{m}^{-2}$; positive $k_1$ focuses horizontally and defocuses vertically. The beam has geometric emittance

$$
\epsilon_x=\epsilon_y=6\ \mathrm{mm\,mrad}=6\times10^{-6}\ \mathrm{m\,rad}.
$$

Horizontal dispersion is written $D_x$; the symbol $\eta$ is reserved for the slip factor. For one particle,

$$
\delta=\frac{\Delta p}{p_0},
$$

while $\sigma_\delta=\mathrm{rms}(\delta)$ describes the momentum spread of the whole beam. Thus $\sigma_\delta=0.001$ means a 0.1% RMS momentum spread. Single-particle maps and tune shifts use $\delta$; RMS beam-size and tune-width formulas use $\sigma_\delta$.


In [ ]:
sigma_delta = 1e-3       # 0.1% rms momentum spread
pipe_radius_m = 0.025    # 2.5 cm
n_sigma_aperture = 1.0   # use 1 for the original simplified rms criterion


# A. Dispersion in a FODO lattice

We start from a 5 m FODO cell whose boundaries are at the centers of the focusing quadrupoles. The reference case has no bends. The modified case places one 10-degree sector bend at the center of each FODO drift sector, for 20 degrees of total bending. The two bend placements and the surrounding drift lengths are mirror-symmetric about `QD`.

In the earlier quadrupole-only transport lab model, momentum dependence was not included. In a real fixed-field lattice, quadrupole focusing is momentum-dependent; that is one source of chromaticity. Here we first add a dipole effect: **horizontal dispersion**, written $D_x$ in the lecture notes. Dispersion describes how the closed-orbit position changes for particles whose momentum differs from the reference momentum. The symbol $\eta$ is reserved for the slip factor.

The physical reason is magnetic rigidity. An off-momentum particle has a different

$$
B\rho = \frac{p}{q},
$$

so the same dipole field bends it by a slightly different amount. Just like $\beta$ and $\alpha$, the dispersion can have a matched periodic solution in a repeating cell.

For the first-order calculations in Sections A and B, the horizontal state is $[x,x',\delta]$. Dipoles supply the source term that generates $D_x$ for off-momentum particles.


## A0. Reference matched FODO cell with no dipole

Run the baseline matched cell. Figure A1 shows its periodic optics and beam size, and Table A1 gives the reference beam sizes at the two focusing-quadrupole halves (`QFa`, `QFb`) and at `QD`. These are the comparison values before adding bends.

The cell below builds the FODO lattice in the same style used in current Xsuite examples: create an `xt.Environment`, define named elements with `env.new(...)`, and assemble an `xt.Line` from a list of component names. The same `QFh` and `D` components are intentionally repeated in the line. The local optics table gives those repeated occurrences names like `QFa`, `D1`, `D2`, and `QFb`.

In [ ]:
env_fodo = xt.Environment()
env_fodo["kq"] = 0.6

env_fodo.new("QFh", xt.Quadrupole, length=0.25, k1="kq")
env_fodo.new("D", xt.Drift, length=2.0)
env_fodo.new("QD", xt.Quadrupole, length=0.5, k1="-kq")

fodo_reference_line = env_fodo.new_line(
    name="fodo_reference",
    components=["QFh", "D", "QD", "D", "QFh"],
)
fodo_reference_line.particle_ref = xt.Particles(p0c=1e9, mass0=xt.ELECTRON_MASS_EV, q0=-1)

fodo_reference, optics_fodo_reference = dch.display_fodo_reference_from_xsuite_line(fodo_reference_line)


## A1. Add dipoles to the two FODO drifts

Now add 20 degrees of total bending while preserving the mirror symmetry of the FODO cell. Each 2.0 m drift sector is replaced by a 0.75 m drift, a 0.5 m Xsuite sector bend, and another 0.75 m drift. The two bends each provide 10 degrees of bending and have edge focusing turned off. Figure A2 shows the resulting periodic optics and dispersion.

In [ ]:
env_bend = xt.Environment()
env_bend["kq"] = env_fodo["kq"]
env_bend["bend_angle"] = np.deg2rad(20.0)
env_bend["bend_angle_each"] = "0.5 * bend_angle"
env_bend["bend_length"] = 0.5

env_bend.new("QFh", xt.Quadrupole, length=0.25, k1="kq")
env_bend.new("D", xt.Drift, length=0.75)
env_bend.new(
    "BEND",
    xt.Bend,
    length="bend_length",
    angle="bend_angle_each",
    k0="bend_angle_each / bend_length",
)
env_bend.new("QD", xt.Quadrupole, length=0.5, k1="-kq")

# Build the FODO cell with two bend occurrences.
fodo_with_bend_line = env_bend.new_line(
    name="fodo_with_bend",
    components=["QFh", "D", "BEND", "D", "QD", "D", "BEND", "D", "QFh"],
)

# Specify particle type.
fodo_with_bend_line.particle_ref = xt.Particles(p0c=1e9, mass0=xt.ELECTRON_MASS_EV, q0=-1)

fodo_with_bend, optics_fodo_bend = dch.display_fodo_bend_from_xsuite_line(fodo_with_bend_line)


**Table A2 — No-bend and symmetric two-bend FODO beam sizes at quadrupole centers ($\sigma_\delta=0$)**

Table A2 makes the direct comparison needed for Q1. It contains only the beam sizes; use Figure A2 to determine the dispersion minimum and its location.


In [ ]:
reference_centers = dch.table_at_element_centers(
    optics_fodo_reference, ["QFa", "QD", "QFb"], sigma_delta=0.0
).set_index("element")
bend_centers = dch.table_at_element_centers(
    optics_fodo_bend, ["QFa", "QD", "QFb"], sigma_delta=0.0
).set_index("element")

table_a2 = pd.DataFrame({
    "element": ["QFa", "QD", "QFb"],
    "σₓ, no bend (mm)": reference_centers.loc[["QFa", "QD", "QFb"], "sigma_x_mm"].to_numpy(),
    "σₓ, two bends (mm)": bend_centers.loc[["QFa", "QD", "QFb"], "sigma_x_mm"].to_numpy(),
    "σᵧ, no bend (mm)": reference_centers.loc[["QFa", "QD", "QFb"], "sigma_y_mm"].to_numpy(),
    "σᵧ, two bends (mm)": bend_centers.loc[["QFa", "QD", "QFb"], "sigma_y_mm"].to_numpy(),
})
print("Table A2: No-bend and symmetric two-bend FODO beam sizes at quadrupole centers")
display(dch.compact_table(table_a2))


## Q1. Minimum dispersion and beam-size comparison

Use **Figure A2** to find the minimum horizontal dispersion $D_x$ in the symmetric two-bend FODO cell. Report its value and identify whether the minimum occurs in a focusing quadrupole, defocusing quadrupole, drift, or bend.

Next use **Table A2**, whose beam sizes are calculated with $\sigma_\delta=0$, to compare the no-bend and two-bend cells. Which horizontal and vertical beam sizes change? Explain why none of these differences is dispersive beam-size growth, even though $D_x$ is nonzero in Figure A2. What property of the sector bends can change the on-momentum betatron optics?


#### Dispersion, momentum spread, and beam size

Horizontal dispersion $D_x(s)$ describes how strongly the horizontal orbit at position $s$ responds to a particle's fractional momentum offset $\delta=\Delta p/p_0$. To first order, the horizontal coordinate can be written

$$
x(s,\delta)=x_\beta(s)+D_x(s)\,\delta.
$$

Thus, $D_x$ is a property of the lattice; it is not itself a beam size. A beam becomes wider from dispersion only when its particles have a range of momenta. That range is described by $\sigma_\delta=\mathrm{rms}(\delta)$. For a beam with no correlation between betatron position and momentum offset, the horizontal rms beam size is

$$
\sigma_x^2 = \epsilon_x\beta_x + D_x^2\sigma_\delta^2.
$$

The first term, $\epsilon_x\beta_x$, is the betatron contribution: $\epsilon_x$ describes the beam and $\beta_x(s)$ describes the lattice focusing. The second term is the additional width caused by momentum spread in a region of nonzero dispersion.

For Table A2, $\sigma_\delta=0$: this idealized beam contains only reference-momentum particles, so every particle has $\delta=0$. The dispersive term therefore vanishes even though Figure A2 shows that the lattice has nonzero $D_x$. This choice isolates changes in the on-momentum optics. Sector bends provide weak horizontal curvature focusing, so inserting them can change $\beta_x$ and therefore $\sigma_x$ through the betatron term alone. That is a betatron-optics change, not dispersive beam-size growth.

In this model there is no vertical dispersion. Therefore, $\sigma_y$ has no dispersive contribution; any vertical size change in Table A2 comes from a change in the vertical betatron optics.


### Figure A3 — Bend-driven off-momentum orbit separation

The colored curves are fixed momentum-offset tracers launched on the same reference orbit and propagated through the lab's first-order model. Move the uniformly spaced $s$ slider: the right panel shows those same tracers at the selected location. Their separation begins in the bends, and the slope of the $x$-versus-$\delta$ slice is the local dispersion. This is an explanatory display; no additional response is required.


In [ ]:
dch.plot_linked_orbit_fan(
    optics_fodo_bend,
    sigma_delta=sigma_delta,
    sample_step_m=0.1,
    title="Figure A3 — Bend-driven off-momentum orbit separation",
)


### Beam size with 0.1% momentum spread

Assume the beam has rms fractional momentum spread

$$
\sigma_\delta = \mathrm{rms}(\delta) = \mathrm{rms}\!\left(\frac{\Delta p}{p_0}\right)=0.001.
$$

This does not mean every particle has $\delta=0.001$; it means the rms width of the beam's $\delta$ distribution is 0.001. Table A3 provides the inputs at `QFa` for the calculation

$$
\sigma_x = \sqrt{\epsilon_x\beta_x + (D_x\sigma_\delta)^2}.
$$

Table A3 supplies the quantities needed to calculate the $\sigma_\delta=0.001$ beam sizes in Q2.


In [ ]:
qfa_inputs = dch.table_at_element_centers(optics_fodo_bend, ["QFa"], sigma_delta=0.0)
table_a3 = dch.compact_table(
    qfa_inputs,
    columns=["element", "beta_x_m", "beta_y_m", "eta_x_m", "sigma_x_mm", "sigma_y_mm"],
    labels={
        "beta_x_m": "βₓ (m)",
        "beta_y_m": "βᵧ (m)",
        "eta_x_m": "Dₓ (m)",
        "sigma_x_mm": "σₓ at σδ=0 (mm)",
        "sigma_y_mm": "σᵧ at σδ=0 (mm)",
    },
)
print("Table A3 — Inputs at QFa for the 0.1% momentum-spread calculation")
display(table_a3)


## Q2. Beam size with 0.1% momentum spread

- Using the values in Table A3, calculate $\sigma_x$ at `QFa` for $\sigma_\delta=0.001$ and compare it with the value of $\sigma_x$ when $\sigma_\delta=0$. 
- Then compare $\sigma_y$ for the same two momentum spreads and explain why the horizontal and vertical sizes behave differently.

**Your Q2 answer**

**Q2 response table — Beam sizes at QFa**

| quantity at QFa | $\sigma_\delta=0$ | $\sigma_\delta=0.001$ |
|---|---:|---:|
| $\sigma_x$ | | |
| $\sigma_y$ | | |

Explanation = 

## Interactive A1 — Bend angle and momentum spread

Move the bend-angle and $\sigma_\delta$ sliders. The linked display keeps the periodic optics, dispersion, and rms beam sizes on fixed axes while the quadrupole strength remains unchanged.

NOTE: This is not used in any question, and is only for your own knowledge.

In [ ]:
dch.interactive_fodo_dispersion()


<details>
<summary><strong>Optional note A1 — Rectangular bends and edge focusing</strong></summary>

The core lab uses symmetric Xsuite sector bends so pole-face edge focusing is not part of the required analysis. A rectangular bend, represented in Xsuite by [`xt.RBend`](https://xsuite.readthedocs.io/en/latest/apireference.html#rbend), has a different pole-face geometry and can change the betatron optics even when $\sigma_\delta=0$. This is distinct from dispersive beam-size growth, which requires a nonzero momentum spread.

Nothing in the required lab depends on rectangular bends; this note is included only for readers who are curious about the distinction.

</details>

# B. Designing an achromatic section

The next model is a compact two-bend achromat-like section with two 18-degree bends. With all quadrupoles off, a beam that starts with zero dispersion does not end with zero dispersion. We will use quadrupoles to make $D_x$ and $D_x'$ return to zero at the section boundary.

Here $D_x'(s)\equiv dD_x/ds$. Because both $D_x$ and $s$ have units of length, $D_x'$ is dimensionless (and is often read as an angle-like quantity). The dispersive contribution to particle slope is $x'_{\mathrm{disp}}\approx D_x'\delta$. An achromatic exit requires both $D_x=0$ and $D_x'=0$.


## B0. Two-bend cell with all quadrupoles off

Use transport optics, not periodic optics, for this first part. The initial conditions are

$$
\beta_x=\beta_y=10\ \mathrm{m},\qquad \alpha_x=\alpha_y=0,
$$

with zero incoming dispersion. Again, the lattice is assembled with Xsuite `Environment` syntax before being converted to the local first-order model used for the achromat exercise.

For Figure B0: no work needed! Observe how the lattice is generated and how with the quadrupoles turned off beta and dispersion increases over the lattice, then proceed on in the lab.

In [ ]:
env_dba_off = xt.Environment()
env_dba_off["bend_angle"] = np.deg2rad(18.0)
env_dba_off["bend_length"] = 1.0

env_dba_off.new("D0", xt.Drift, length=0.5)
env_dba_off.new("Q2", xt.Quadrupole, length=0.3, k1=0.0)
env_dba_off.new("D1", xt.Drift, length=0.5)
env_dba_off.new("B1", xt.Bend, length="bend_length", angle="bend_angle", k0="bend_angle / bend_length")
env_dba_off.new("D2", xt.Drift, length=2.3)
env_dba_off.new("Q1", xt.Quadrupole, length=0.3, k1=0.0)
env_dba_off.new("D3", xt.Drift, length=2.3)
env_dba_off.new("B2", xt.Bend, length="bend_length", angle="bend_angle", k0="bend_angle / bend_length")
env_dba_off.new("D4", xt.Drift, length=0.5)
env_dba_off.new("Q3", xt.Quadrupole, length=0.3, k1=0.0)
env_dba_off.new("D5", xt.Drift, length=0.5)
two_bend_off_line = env_dba_off.new_line(
    name="two_bend_off",
    components=["D0", "Q2", "D1", "B1", "D2", "Q1", "D3", "B2", "D4", "Q3", "D5"],
)
two_bend_off_line.particle_ref = xt.Particles(p0c=1e9, mass0=xt.ELECTRON_MASS_EV, q0=-1)

two_bend_off, optics_two_bend_off, Dx_end_off, Dxp_end_off = dch.display_dba_insert_from_xsuite_line(two_bend_off_line)


### Figure B1 — The two achromat conditions as one state

The left panels show $D_x$ and $D_x'$. The right panel follows the same state through the line as s progresses. An achromatic exit must return to the open-circle target at the origin. The exit marker shows whether the uncorrected section finishes instead.


In [ ]:
dch.plot_dispersion_state_portrait(
    optics_two_bend_off,
    title="Figure B1 — Dispersion state through the uncorrected two-bend section",
)


## B1. Tune Q1 to close the achromat

The central quadrupole Q1 changes the dispersion trajectory between the two bends. An achromatic exit requires both $D_x(s_\mathrm{end})=0$ and $D_x'(s_\mathrm{end})=0$, so the line must return to the origin in $(D_x,D_x')$ state space.

In a general lattice, one knob cannot satisfy two independent exit conditions. Here the two bends and surrounding drifts are mirror-symmetric about the center of Q1: when Q1 makes $D_x'=0$ at that symmetry plane, the second half reverses the first and returns the incoming $(D_x,D_x')=(0,0)$ state to the origin. In **Interactive B1**, achromat closure is indicated when the line-exit marker approaches the open-circle target at $(0,0)$.


### Interactive B1 — Tune Q1 to close the achromat

The left panels show $D_x$ and $D_x'$. The right panel follows the same state through the line as s progresses. An achromatic exit must return to the open-circle target at the origin. The exit marker shows whether the uncorrected section finishes instead.

Move the **Q1 $k_1$** slider and watch how the dispersion state changes through the two bends. In the right-hand **dispersion-state portrait**, the open circle is the achromat target $(D_x,D_x')=(0,0)$ and the × marker is the state at the line exit. 

In [ ]:
dch.interactive_achromat_q1()



## Q3. / Q4. End-of-cell dispersion before correction

In Interactive Figure B1...

Q3) Start with Q1 k1 = 0. What are the final uncorrected values of $D_x(s)$ and $D_x'(s)$? Does this cell qualify as achromatic at its exit?

Q4) Adjust Q1 until the exit marker lies as close as possible to the target. What value of Q1 k1 is required to close the achromat. Report the best Q1 setting to the nearest $0.05\ \mathrm{m}^{-2}$.

In [ ]:
# Refine the Q1 strength for the downstream calculations. Q4 itself is answered
# at the interactive slider resolution. The helpers reuse the DBA element order
# and lengths constructed explicitly with Xsuite above.
q1_achromat = dch.best_q1_for_achromat(qmin=0.0, qmax=6.0)


## B2. Add focusing so the cell can be repeated

Canceling outgoing dispersion in an open beamline does not automatically make a stable periodic cell. The default focused double-bend achromat-like (DBA-like) cell keeps the same Q1 achromat condition and adds two flanking quadrupoles with fixed strengths

$$
k_{1,\mathrm{Q2}}=+2.475\ \mathrm{m}^{-2},\qquad
k_{1,\mathrm{Q3}}=-2.150\ \mathrm{m}^{-2}.
$$

Their placement is outside the achromatic section's zero-dispersion boundaries, so they can change periodic focusing while preserving the local closure condition. These are supplied baseline strengths, not additional quantities to solve for.


In [ ]:
# Fixed baseline strengths for the focused DBA-like cell used below.
q2_focused = 2.475
q3_focused = -2.150
central_q1_cell, focused_dba_cell, optics_focused_dba = dch.display_focused_dba_cell(
    q1_achromat, q2=q2_focused, q3=q3_focused
)

# Separate, denser data product for the numerical extrema and aperture checks below.
# The ordinary Figure B2 optics remain at the lighter plotting density.
optics_focused_dba_diagnostics = dch.compute_periodic_optics(
    focused_dba_cell,
    samples_per_meter=dch.DIAGNOSTIC_SAMPLES_PER_METER,
)


## Q5. Maximum dispersion in the focused DBA-like cell

Use **Figure B2** to locate the internal dispersion peak. Report the maximum $D_x$ in the stable focused cell and its location. Explain why a large internal bump is compatible with achromatic entrance and exit boundaries.


**Your Q5 answer**

Maximum $D_x$ =  
Location =  
Why this is compatible with achromatic boundaries:


In [ ]:
q5_extrema = dch.dispersion_extrema(optics_focused_dba_diagnostics)
print("Table B1 — Numerical check of the maximum-dispersion location")
display(dch.compact_table(
    q5_extrema.query("condition == 'maximum D_x'"),
    columns=["element", "s_m", "eta_x_m", "eta_xp"],
    precision=4,
))


## Horizontal aperture-envelope proxy

Because the effect of dispersion depends on momentum spread, a beam that fits on momentum can develop a larger horizontal RMS envelope when $|D_x|\sigma_\delta$ is large. Here $\sigma_\delta$ is the RMS width of the beam's $\delta=\Delta p/p_0$ distribution.

With a finite pipe radius, momentum spread can make the horizontal envelope too large. Here $n_\sigma$ is the number of horizontal RMS beam widths included in the envelope: $n_\sigma=1$ compares one RMS width with the pipe radius, while larger values impose a stricter margin. The simplified criterion used here is

$$
n_\sigma\sqrt{\epsilon_x\beta_x+D_x^2\sigma_\delta^2}=R.
$$

This condition is checked along the cell. The controlling location is not necessarily the maximum-$|D_x|$ point because the betatron contribution $\epsilon_x\beta_x$ also matters. If $n_\sigma\sqrt{\epsilon_x\beta_x}\ge R$ already at $\sigma_\delta=0$, then that criterion has **no on-momentum clearance** and its momentum-spread allowance is zero; a zero-dispersion location must not be reported as having infinite allowance in that case.

The original lab effectively used the $n_\sigma=1$ horizontal RMS proxy. This is not particle-survival tracking, and it does not test the vertical aperture. You can later change `n_sigma_aperture` to inspect stricter envelope conventions.


## Q6. Momentum-spread limit from a 2.5 cm pipe radius

Using $R=2.5\ \mathrm{cm}$ and $n_\sigma=1$, use the $\beta_x$ and $D_x$ inputs in **Table B2** with the equation above to calculate the largest allowed $\sigma_\delta$. Report the result as a fraction and a percent.


In [ ]:
aperture_limits = dch.aperture_limit_table(
    optics_focused_dba_diagnostics,
    pipe_radius_m=pipe_radius_m,
    n_sigma=n_sigma_aperture,
)
dispersion_limit = float(aperture_limits.iloc[0]["max_sigma_delta"])
dispersion_limit_location = str(aperture_limits.iloc[0]["element"])

print("Table B2 — Optics inputs at the controlling aperture location")
display(dch.compact_aperture_input_row(aperture_limits, precision=4))


**Your Q6 answer**

Largest allowed $\sigma_\delta$ =  
Equivalent percent momentum spread =  
Controlling element/location =


## B3. Visual check just above the Q6 envelope limit

Figure B3 uses $\sigma_\delta=1.10\,\sigma_{\delta,\mathrm{lim}}$ from Q6. It separates the betatron, dispersive, and quadrature-total envelopes, then shows the remaining clearance to the pipe. The zero-clearance line lets you identify crossings from the curves without marked coordinates. This is an ungraded visual conclusion, not a particle-loss simulation.

In [ ]:
# Change this variable! ↓
sigma_delta_test = 1.1 * dispersion_limit

dch.plot_aperture_ribbon(
    optics_focused_dba_diagnostics,
    sigma_delta=sigma_delta_test,
    pipe_radius_m=pipe_radius_m,
    n_sigma=n_sigma_aperture,
    title=f"Figure B3 — Envelope components and clearance at 1.10× the Q6 limit",
)

### What Figure B3 shows

The faint tracked distribution makes the orbit spread concrete, while the three envelope curves show how betatron width and dispersive width combine in quadrature. Wherever the clearance curve falls below zero, the selected rms-envelope criterion exceeds the pipe.

## Try it: aperture and momentum spread

Use the sliders to compare different momentum spreads and horizontal aperture criteria, especially $n_\sigma=1$, 3, and 5. If the total envelope already lies above the pipe at $\sigma_\delta=0$, the additional momentum-spread allowance is zero for that criterion. The axes stay fixed; curves that leave the vertical range have exceeded the displayed pipe-scale view.

In [ ]:
dch.interactive_aperture(optics_focused_dba_diagnostics)


# C. Chromaticity in a ring

Now repeat the stable DBA-like cell 10 times to form a model ring. Each cell bends through $2\times18^\circ=36^\circ$, so ten cells give the required $360^\circ$ total bend. The tune is the total betatron phase advance divided by $2\pi$. The lecture notes write the tune as $\nu_u$; this notebook uses the equivalent symbol $Q_u\equiv\nu_u$ to match Xsuite's `qx` and `qy` output names:

$$
Q_u=\frac{1}{2\pi}\oint \frac{ds}{\beta_u(s)}.
$$

Off-momentum particles see different effective focusing, so their tunes shift with the individual particle momentum offset $\delta=\Delta p/p_0$. This lab uses the unnormalized chromaticity convention

$$
\xi_u=\left.\frac{dQ_u}{d\delta}\right|_{\delta=0}.
$$

The DBA-like cell has regions where the dispersion is locally close to zero. That is useful: off-momentum particles do not get a large dispersive beam-size contribution in those regions. But zero dispersion in part of the lattice does **not** mean the ring has no chromaticity.

Chromaticity comes from the momentum dependence of focusing. An off-momentum particle is not bent and focused exactly like the reference particle, so it oscillates with a slightly different tune. Section C uses Xsuite directly for the periodic ring optics, chromaticities, and off-momentum tune scans.


In [ ]:
n_ring_cells = 10
ring_cell = focused_dba_cell
ring_xsuite = dch.xsuite_ring_from_lattice(
    ring_cell,
    n_cells=n_ring_cells,
    particle_ref=two_bend_off_line.particle_ref,
)

# this cell does not display anything. it computes the ring optics and checks the periodicity of the Twiss functions.


## Nominal ring tunes supplied by Xsuite

Run the direct Xsuite Twiss calculation below. It supplies the nominal tunes $Q_x=5.934679$ and $Q_y=2.599888$ as setup for the interpretive questions that follow; recording or rounding them is not a separate exercise.


In [ ]:
ring_twiss = ring_xsuite.twiss(method="4d", chrom=True, delta_chrom=1e-4)
Qx, Qy = float(ring_twiss.qx), float(ring_twiss.qy)
print(f"Qx = {Qx:.6f}")
print(f"Qy = {Qy:.6f}")


## Q7. Why are the tunes not necessarily equal?

In the earlier no-bend FODO reference case, the horizontal and vertical phase advances were equal because the lattice was symmetric between the two planes. In Figure C1, the lower panel is the local slope $d(\psi/2\pi)/ds$ of the accumulated curves above it. Compare where the horizontal and vertical phase advance accumulates most rapidly, then explain why equal tunes should not be expected in the DBA-like cell.

In [ ]:
import plotly.graph_objects as go

ring_twiss_dense_first_cell = dch.xsuite_dense_twiss_for_first_cell(
    ring_xsuite,
    ring_cell,
    n_points=241,
)
first_cell_twiss = ring_twiss_dense_first_cell.to_pandas()
first_cell_length = sum(element.length for element in ring_cell)
first_cell_twiss = first_cell_twiss.loc[first_cell_twiss["s"] <= first_cell_length + 1e-12]

beta_fig = go.Figure()
beta_fig.add_trace(go.Scatter(
    x=first_cell_twiss["s"], y=first_cell_twiss["betx"],
    mode="lines", name="βₓ",
    hovertemplate="βₓ = %{y:.5g} m<extra></extra>",
))
beta_fig.add_trace(go.Scatter(
    x=first_cell_twiss["s"], y=first_cell_twiss["bety"],
    mode="lines", name="βᵧ",
    hovertemplate="βᵧ = %{y:.5g} m<extra></extra>",
))
beta_fig.update_layout(
    title="Beta functions corresponding to Figure C1 — smaller β means faster phase accumulation",
    xaxis_title="s [m]", yaxis_title="β [m]",
    template="plotly_white", hovermode="x unified", height=430,
)
beta_fig.update_xaxes(unifiedhovertitle=dict(text="s = %{x:.3f} m"))
dch.add_lattice_strip(beta_fig, dch.element_layout(ring_cell))
beta_fig.show()

dch.plot_xsuite_accumulated_phase_advance(
    ring_twiss_dense_first_cell,
    ring_cell,
    title="Figure C1 — Accumulated phase advance and its local accumulation rate",
)


**Your Q7 answer**

Reason the horizontal and vertical tunes need not be equal:


## Q8. Chromaticity and tune width for 0.1% momentum spread

For one particle, the linearized signed tune shift is

$$
\Delta Q_u(\delta)=\xi_u\delta.
$$

For a centered beam with rms momentum spread $\sigma_\delta$, the corresponding rms tune width is nonnegative:

$$
\sigma_{Q_u}=|\xi_u|\sigma_\delta.
$$

Use the three emphasized Xsuite points at $\delta=-10^{-4},0,+10^{-4}$ in Figure C2 and the centered finite difference

$$
\xi_u\approx\frac{Q_u(+\Delta\delta)-Q_u(-\Delta\delta)}{2\Delta\delta}
$$

to determine $\xi_x$ and $\xi_y$. Then calculate $\sigma_{Q_x}$ and $\sigma_{Q_y}$ for $\sigma_\delta=0.001$.


In [ ]:
# Xsuite evaluates the same ring directly at every displayed momentum offset.
xi_x, xi_y = float(ring_twiss.dqx), float(ring_twiss.dqy)
delta_scan = np.unique(np.r_[
    np.linspace(-0.01, 0.01, 81),
    [-1e-4, 0.0, 1e-4],
])
xsuite_tune_scan = dch.xsuite_tune_scan(ring_xsuite, delta_values=delta_scan)
dch.plot_tune_scan_and_footprint(
    xsuite_tune_scan,
    sigma_delta=sigma_delta,
    delta_range=(-2.0 * sigma_delta, 2.0 * sigma_delta),
    ddelta=1e-4,
    resonance_order=3,
    title="Figure C2 — Xsuite tune scan and resonance footprint",
)

**Your Q8 answer**

$\xi_x=$  
$\xi_y=$  
$\sigma_{Q_x}$ for $\sigma_\delta=0.001$ =  
$\sigma_{Q_y}$ for $\sigma_\delta=0.001$ =


## Resonance diagram and chromatic tune footprint

A resonance line satisfies

$$
mQ_x+nQ_y=p,
$$

where $m$, $n$, and $p$ are integers. The resonance order is $|m|+|n|$. In this exercise, assume resonances of order 3 and below are forbidden, while order 4 and above are tolerable. **Figure C2** is the fixed tune-scan overview used in Q8; the variable-width resonance estimate in Q9 uses **Interactive C1** below.


## Interactive C1 — Resonance margin from the chromatic tune footprint

Keep the resonance order at 3 and increase $\sigma_\delta$ until the Xsuite tune footprint first touches a resonance line of order 3 or lower. Hover over that line to identify its equation. The displayed footprint spans $-\sigma_\delta\le\delta\le+\sigma_\delta$, so the first crossing at $|\delta_\mathrm{cross}|$ defines the corresponding threshold $\sigma_\delta$ proxy. Record your visual estimate before running the Table C1 check in Q9.


In [ ]:
dch.interactive_tune_footprint_from_scan(xsuite_tune_scan)


## Q9. Resonance-margin proxy from Interactive C1

Use **Interactive C1 immediately above**—not the fixed $\sigma_\delta=0.001$ overview in Figure C2—to estimate the largest $\sigma_\delta$ for which the direct Xsuite tune footprint remains clear of a resonance through order 3. Identify the first resonance touched, then compare your graphical estimate with **Table C1**, produced by the scan-based check below. This is a resonance-margin proxy, not a tracked hard momentum-acceptance boundary.


**Your Q9 answer**

Largest tolerable $\sigma_\delta$ from Interactive C1 =  
Nearest forbidden resonance =


In [ ]:
resonance_limits = dch.first_resonance_crossing_from_tune_scan(
    xsuite_tune_scan,
    max_order=3,
)
print("Table C1 — Three nearest order-3-or-lower resonance crossings")
display(dch.compact_resonance_crossing(resonance_limits.head(3)))

resonance_margin_delta = float(resonance_limits.iloc[0]["abs_delta_at_crossing"])
nearest_resonance = str(resonance_limits.iloc[0]["resonance"])


## Q10. Compare the two simplified limits

Compare the aperture-envelope proxy from Q6 with the Xsuite resonance-margin proxy from Q9. Which mechanism gives the smaller momentum-spread scale, and why? Neither proxy is a nonlinear survival-tracking calculation.


**Your Q10 answer**

The smaller momentum-spread proxy is:  
Reason:


In [ ]:
print("Table C2 — Comparison of aperture and resonance-margin proxies")
comparison = dch.compact_acceptance_comparison(
    dispersion_limit_delta=dispersion_limit,
    chromatic_limit_delta=resonance_margin_delta,
)
display(comparison)


# Optional extension exercises

1. Repeat Q6 with `n_sigma_aperture = 3`. Does the aperture limit become more or less restrictive than the chromaticity limit?
2. Change the FODO bend angle in Section A. How does $D_x$ scale with bend angle for small angles?
